# Лексический поиск: от подстроки к поисковому сервису

**Цель:** построить поиск по отзывам и понять, чем отличаются совпадение подстроки, TF-IDF и поисковый индекс.

После занятия вы сможете:

- представить документы и запрос в одном пространстве TF-IDF;
- объяснить оценки косинусного сходства и обработать пустую выдачу;
- создать индекс ZincSearch, загрузить документы и прочитать результаты поиска;
- сравнить методы на одном корпусе и оценить релевантность top-k.

**План:** подготовка → данные → подстрока → TF-IDF → ZincSearch → сравнение и оценка.

В основном сценарии все методы работают на **одной выборке из первых 1000 записей после удаления повторов ID**. Массовая загрузка полного датасета вынесена в дополнительный раздел.

## 1. Подготовка окружения

Основной сценарий — локальный запуск из корня репозитория. Нужны Python-окружение с зависимостями из `requirements.txt`, Docker и файл `.env`. Учётные данные задаются через `ZINCSEARCH_USERNAME` и `ZINCSEARCH_PASSWORD`.

```shell
make run-jupyter
```

Откройте адрес Jupyter из терминала (порт 8899). Для части про ZincSearch запустите в другом терминале:

```shell
make run-search
```

ZincSearch доступен по адресу `http://localhost:4080`. Здесь запускается только поисковый движок; FastAPI и Telegram-бот для занятия не нужны. В Google Colab можно выполнить часть про TF-IDF после настройки доступа к данным и `src`, но локальный Docker-сервис из Colab по адресу `localhost` недоступен.

Скачайте [Health_and_Personal_Care.jsonl.gz](https://drive.google.com/file/d/12N52kB4D1iqgzSuoWEfNSY3KqVRp10wL/view?usp=drive_link) и поместите в `data` в корне репозитория. Это отзывы на товары: мы ищем тексты, а не проверяем достоверность утверждений авторов отзывов.

In [ ]:
import os
import sys
from pathlib import Path

# Jupyter может открыться в корне репозитория или в jupyter_notebooks.
current_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / 'src' / 'utils.py').is_file() and (path / 'configs').is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError('Откройте ноутбук из репозитория ai_product_engineer.')
sys.path.insert(0, str(repo_root / 'src'))

root_data_dir = repo_root / 'data'
data_path = root_data_dir / 'Health_and_Personal_Care.jsonl.gz'
env_path = repo_root / '.env'
os.environ['CONFIG_DIR'] = str(repo_root / 'configs')
# Ядро Jupyter работает на компьютере, ZincSearch опубликован на порту 4080.
os.environ['ZINCSEARCH_URL'] = 'http://localhost:4080'

if not data_path.is_file():
    raise FileNotFoundError(f'Скачайте датасет в {data_path}')
print(f'Данные: {data_path}')

## 2. Знакомство с данными

Одна запись — один отзыв. Поле `text` содержит его текст, `asin` и `parent_asin` — идентификаторы товара, `user_id` и `timestamp` помогут построить стабильный ID документа.

Для интерактивного занятия читаем первые 1000 записей и оставляем одну запись на ID, сохраняя последнюю при повторе, как при обновлении документа в индексе. Итоговый корпус может быть немного меньше 1000 документов. Число отзывов и число уникальных товаров различаются: у товара может быть несколько отзывов.

In [ ]:
import pandas as pd
from utils import read_raw_data, eval_doc_id

SAMPLE_SIZE = 1000
raw_sample = read_raw_data(str(data_path), limit=SAMPLE_SIZE)
# ZincSearch хранит один документ на _id: приводим TF-IDF к тому же корпусу.
json_data = list({
    eval_doc_id(item['user_id'] + str(item['timestamp'])): item
    for item in raw_sample
}.values())
print(f'Повторов ID в исходной выборке: {len(raw_sample) - len(json_data)}')
df = pd.json_normalize(json_data)
assert not df.empty, 'Выборка пуста.'
assert df['text'].notna().all(), 'В выборке есть отзывы без текста.'
print(f'Отзывов: {len(df)}, уникальных товаров: {df["asin"].nunique()}')
df[['title', 'text', 'asin']].head(3)

## 3. Базовый метод: поиск подстроки

Найдём отзывы, содержащие `cough`. Приводим запрос и документ к нижнему регистру. Этот метод отвечает только на вопрос о наличии подстроки: он не ранжирует результаты по полезности и может находить часть более длинного слова.

In [ ]:
user_query = 'cough'
normalized_query = user_query.strip().lower()
substring_results = (
    df['text'].str.lower().str.contains(normalized_query, regex=False)
    if normalized_query else pd.Series(False, index=df.index)
)
print(f'Совпадений: {int(substring_results.sum())} из {len(df)}')
df.loc[substring_results, ['title', 'text']].head(5)

## 4. TF-IDF: взвешенные совпадения слов

**TF (term frequency)** отражает частоту слова в документе. **IDF (inverse document frequency)** уменьшает вес слов, которые встречаются во многих документах корпуса. Их произведение помогает выделять слова, характерные для конкретного текста.

`TfidfVectorizer.fit` строит словарь и вычисляет статистику корпуса; это не обучение на размеченной релевантности. `transform` переводит тексты в векторы с этим словарём и весами.

В матрице ниже **строки — документы, столбцы — термины**, значения — веса. Матрица разреженная: большинство документов содержит лишь небольшую часть словаря.

Выбранная токенизация оставляет слова длиной от трёх символов. Например, `mg` будет отброшено. Это решение влияет на поиск, а не просто ускоряет вычисления.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer='word',
    lowercase=True,
    token_pattern=r'\b[\w\d]{3,}\b',
)
vectorizer.fit(df['text'])
document_vectors = vectorizer.transform(df['text'])
print(f'Матрица: {document_vectors.shape[0]} документов × {document_vectors.shape[1]} терминов')
print(f'Ненулевых элементов: {document_vectors.nnz}')
print('Пример терминов:', vectorizer.get_feature_names_out()[:15])

### Ранжирование по косинусному сходству

Запрос преобразуем **тем же vectorizer**: координаты запроса и документов должны означать одни и те же слова.

Косинусное сходство: `cos(A, B) = (A · B) / (||A|| × ||B||)`. Оно не меняется при умножении ненулевого вектора на положительное число. Для неотрицательных TF-IDF-векторов значение находится в `[0, 1]`. Высокий score означает близость в выбранном представлении, а не гарантированную релевантность.

`TfidfVectorizer` по умолчанию применяет L2-нормализацию. Для таких ненулевых векторов косинусное сходство совпадает со скалярным произведением. Не нужно утверждать, что евклидово расстояние всегда хуже: на единичных векторах оно связано с косинусным сходством формулой `||A − B||² = 2 − 2cos(A, B)`.

**Top-k — верхняя граница количества результатов.** Если положительных совпадений три, выдача из десяти документов добавит семь документов с нулевым score. Поэтому ниже они отфильтрованы.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_search(query, k=10):
    if not query.strip():
        return pd.DataFrame(columns=['row_id', 'score', 'title', 'text'])
    query_vector = vectorizer.transform([query])
    if query_vector.nnz == 0:
        return pd.DataFrame(columns=['row_id', 'score', 'title', 'text'])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    positive = np.flatnonzero(scores > 0)
    top_indices = positive[np.argsort(-scores[positive], kind='stable')[:k]]
    result = df.iloc[top_indices][['title', 'text']].copy()
    result.insert(0, 'score', scores[top_indices])
    result.insert(0, 'row_id', top_indices)
    return result.reset_index(drop=True)

tfidf_results = tfidf_search(user_query)
print(f'Результатов с положительным score: {len(tfidf_results)}')
tfidf_results

### Проверяем границы метода

Пустой запрос и запрос без терминов из словаря должны вернуть пустую выдачу. Лексический поиск может пропустить синоним без общих слов и не различить полезный совет, отрицание и случайное упоминание темы.

**Задание:** сравните `cough`, `dry cough` и `coughing`. Объясните изменения через токенизацию и веса слов. Затем попробуйте символьные n-граммы (`analyzer='char_wb'`, `ngram_range=(3, 5)`) и сравните результаты, включая нерелевантные совпадения.

In [ ]:
assert tfidf_search('').empty
assert tfidf_search('zzzzunseenquerytokenzzzz').empty
for query in ['cough', 'dry cough', 'coughing']:
    result = tfidf_search(query, k=3)
    print(f'{query!r}: {len(result)} результатов')
    print(result[['row_id', 'score', 'title']].to_string(index=False))
    print()

## 5. Зачем поисковый сервис

Локальный пример хранит матрицу в памяти Python и сравнивает запрос со всеми документами. Поисковый движок отдельно хранит индекс, предоставляет HTTP API, фильтры и выдачу с метаданными.

Здесь используется **ZincSearch**, а не Elasticsearch. Они имеют отдельные реализации и частично совместимые API. Для полнотекстового поиска важен **инвертированный индекс**: по термину можно найти документы, где он встречается. Это отличается от индекса ближайших соседей для плотных эмбеддингов.

```text
Подготовка: отзывы → очистка → документы → индекс ZincSearch
Запрос:     текст → поисковый API → ранжированные документы
```

Сравнивать абсолютные score разных движков и TF-IDF нельзя: формулы и статистика могут различаться. Важно сравнивать релевантность результатов на одинаковых данных.

### Подключение

Запустите `make run-search`, если ещё не сделали этого. Параметры авторизации берём из `.env`; сам объект с паролем в вывод не печатаем.

In [ ]:
from utils import get_auth, request_elastic, load_config, create_index

if not env_path.is_file():
    raise FileNotFoundError(f'Создайте {env_path} и настройте учётные данные ZincSearch.')
auth_config = get_auth(env_path)
version = request_elastic('version', conf=auth_config)
if version is None:
    raise RuntimeError('Проверьте make run-search, адрес и учётные данные ZincSearch.')
print('ZincSearch:', version['version'])

### Создание отдельного учебного индекса

В конфигурации `configs/index_config.yml` описаны поля: `content` индексируется как текст, `category` — как точное значение (`keyword`). Анализатор превращает текст в термины; от его настроек зависит, что считается совпадением.

Для каждого запуска создаём уникальный индекс, чтобы выдача не зависела от старых загрузок. Запомните напечатанное имя: оно понадобится для CLI. Повторное выполнение этой ячейки создаёт ещё один индекс; предыдущие автоматически не удаляются.

In [ ]:
from uuid import uuid4

index_config = load_config()
index_name = f"search_lesson_{uuid4().hex[:12]}"
index_config['name'] = index_name
created = create_index(index_config, conf=auth_config)
if created is None:
    raise RuntimeError('Индекс не создан: проверьте ответ ZincSearch выше.')
print('Учебный индекс:', index_name)
print('Поля:', list(index_config['mappings']['properties']))

### Подготовка той же выборки отзывов

Не перечитываем полный датасет: TF-IDF и ZincSearch должны искать по одному корпусу. Стабильный `_id` позволяет повторно загрузить документ с тем же ID без создания отдельной копии. В учебном коде ID вычисляется функцией `eval_doc_id` из пользователя и времени отзыва. Это короткий хеш: при росте корпуса возможны коллизии, поэтому для production нужен более надёжный ключ. Одинаковый способ подготовки документов используется в ноутбуке и CLI.

In [ ]:
from utils import clean_text, review_to_document

docs = [review_to_document(item) for item in json_data]
assert len({doc['_id'] for doc in docs}) == len(docs), 'В выборке есть повторяющиеся ID.'
print(f'Подготовлено документов: {len(docs)}')
print({key: value for key, value in docs[0].items() if key != 'content'})
print('Текст:', docs[0]['content'][:200])

### Пакетная загрузка и проверка готовности поиска

Загружаем по 500 документов в запросе через `POST /api/_bulkv2`, до двух запросов параллельно. Прогресс показывает документы в завершённых пакетах; число ошибок проверяем отдельно. Окончание HTTP-запроса и доступность документа для поиска могут не совпадать по времени, поэтому ниже ждём появления всей выборки в поисковой выдаче.

In [ ]:
from utils import load_bulk_documents, search

upload_result = load_bulk_documents(
    index_name, docs, conf=auth_config, workers=2, batch_size=500,
)
assert upload_result['errors'] == 0, 'Загрузка завершилась с ошибками: проверьте сообщения выше.'

import time

deadline = time.monotonic() + 30
while True:
    response = search(index_name, query=None, limit=1, conf=auth_config)
    if response is None:
        raise RuntimeError('Не удалось проверить доступность документов для поиска.')
    total = response['hits']['total']
    indexed_count = total['value'] if isinstance(total, dict) else total
    if indexed_count == len(docs):
        break
    if time.monotonic() >= deadline:
        raise TimeoutError(f'Для поиска доступно {indexed_count} из {len(docs)} документов.')
    time.sleep(0.5)
print(f'В индексе доступны все {indexed_count} документов.')

### Чтение ответа поиска

Обёртка `search` отправляет текстовый запрос и фильтр `category=health`. В ответе `hits.total` — количество найденных документов, `hits.hits` — выбранные результаты. У результата `_id` обозначает документ, `_score` — оценку движка, `_source` — исходные поля.

Сравните выдачу с TF-IDF: совпадают ли документы в начале списка? Различие само по себе не означает ошибку: движок может иначе анализировать текст и ранжировать совпадения.

In [ ]:
res = search(index_name, user_query, limit=10, conf=auth_config)
if res is None:
    raise RuntimeError('Поисковый запрос завершился ошибкой.')
zinc_hits = res['hits']['hits']
print('Всего совпадений:', res['hits']['total'])
zinc_results = pd.DataFrame([
    {'doc_id': hit['_id'], 'score': hit['_score'],
     'asin': hit['_source']['asin'], 'text': hit['_source']['content'][:300]}
    for hit in zinc_hits
])
zinc_results

## 6. Как оценивать качество поиска

Наличие слова и высокий score ещё не означают полезность результата. Для запроса `cough` отзыв может описывать товар от кашля, а может использовать слово как шутливую вставку.

Сформулируйте задачу точнее: **«найти отзывы с описанием использования товара для облегчения кашля»**. Затем вручную отметьте релевантные документы. Это оценка соответствия текста запросу, а не медицинской достоверности отзыва.

**Precision@k** — доля релевантных документов в первых `k` позициях. Здесь используем фиксированный знаменатель `k`: если результатов меньше, недостающие позиции не добавляют релевантных документов. В системах с другой договорённостью знаменатель может отличаться — это нужно оговаривать.

Чтобы честно сравнить методы, разметьте объединение их top-k для нескольких запросов. Для Recall@k нужен более полный набор известных релевантных документов; одной оценки верхушки выдачи недостаточно.

In [ ]:
def precision_at_k(result_ids, relevant_ids, k=5):
    if k < 1:
        raise ValueError('k должно быть положительным')
    return sum(doc_id in relevant_ids for doc_id in result_ids[:k]) / k

# Мини-пример с заранее заданной разметкой, не измерение качества нашего индекса.
assert precision_at_k(['a', 'b', 'c'], {'a', 'c'}, k=3) == 2 / 3
assert precision_at_k([], {'a'}, k=3) == 0

k = 5
tfidf_ids = [docs[int(row_id)]['_id'] for row_id in tfidf_search(user_query, k)['row_id']]
zinc_ids = [hit['_id'] for hit in zinc_hits[:k]]
by_id = {doc['_id']: doc for doc in docs}
candidates = list(dict.fromkeys(tfidf_ids + zinc_ids))
# Прочитайте кандидатов и составьте relevant_ids вручную перед подсчётом метрик.
pd.DataFrame([
    {'doc_id': doc_id, 'text': by_id[doc_id]['content']}
    for doc_id in candidates
])

### Самостоятельная работа

1. Для трёх запросов разметьте релевантность объединённой выдачи TF-IDF и ZincSearch. Посчитайте `precision_at_k(tfidf_ids, relevant_ids, k)` и аналогично для ZincSearch.
2. Найдите пример совпадения подстроки, которое не является нужным словом, и пример синонима, который не находится лексическим методом.
3. Измените токенизацию TF-IDF. Зафиксируйте, какие запросы улучшились и какие ухудшились.
4. Измерьте время запросов несколько раз на одном корпусе. Отделите время подготовки индекса от времени поиска; первый запрос может отличаться от последующих.
5. Объясните, что произойдёт при повторной загрузке тех же документов с теми же `_id`.

## 7. Дополнительно: загрузка из консоли

Вместо запуска ячейки загрузки можно использовать CLI из `src/utils.py`. В другом терминале **из корня репозитория** выполните команду, подставив имя индекса, напечатанное выше, вместо `ИМЯ_УЧЕБНОГО_ИНДЕКСА`:

```shell
uv run python src/utils.py bulk-search \
  --data-path data/Health_and_Personal_Care.jsonl.gz \
  --env-file .env \
  --url http://localhost:4080 \
  --index ИМЯ_УЧЕБНОГО_ИНДЕКСА \
  --limit 1000 \
  --workers 2 \
  --batch-size 500
```

`--limit 1000` сохраняет тот же корпус, что в ноутбуке. Без `--index` имя берётся из `configs/index_config.yml`. Индекс с нужными mappings следует создать заранее.

- `--workers` — число параллельных запросов (по умолчанию 1).
- `--batch-size` — документов в запросе (по умолчанию 500).
- Очередь вместе с выполняемыми задачами содержит не более `2 × workers` пакетов.
- Прогресс считается по документам; ошибки дают ненулевой код завершения команды.
- Для небольшого пробного запуска используйте отдельный индекс и `--limit 100`.

Чтобы загрузить **весь датасет**, уберите `--limit` и используйте отдельный индекс: сравнение с TF-IDF по 1000 отзывов после этого уже некорректно. Текущий CLI сначала читает отзывы в память; ограниченная очередь запросов не делает чтение датасета потоковым. Большой корпус потребует больше памяти и времени.

Справка: `uv run python src/utils.py bulk-search --help`. Протокол: [ZincSearch Bulk V2](https://zincsearch-docs.zinc.dev/api/document/bulkv2/).

## Итоги

- Поиск подстроки проверяет буквальное совпадение и не ранжирует выдачу.
- TF-IDF учитывает статистику слов в корпусе; запрос и документы должны использовать общий словарь.
- Top-k не гарантирует релевантность: нужны обработка пустой выдачи и оценка результатов.
- ZincSearch хранит отдельный поисковый индекс и предоставляет HTTP API.
- Сравнивать методы нужно на одинаковом корпусе по заданным критериям релевантности.

Далее — [семантический поиск и RAG](vol_02_ml_products_02_search_rag.ipynb): как находить смысловые совпадения без общих слов.

Документация: [TF-IDF в scikit-learn](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting), [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), [API ZincSearch](https://zincsearch-docs.zinc.dev/api/).